In [2]:
import sys, os, time
import jax
import jax.numpy as jnp
from jax import random as jr

jax.config.update("jax_enable_x64", True)

sys.path.append(os.path.dirname(os.getcwd()))
from riemann_pinn.physics import find_pstar, GAMMA

## Primitive states + Euler flux

The project's gas state `(drho, dp, du)` only encodes the velocity *difference*
`uRL = uR - uL = ducrit * du`. The interface flux of an approximate solver
depends on the lab frame, so we adopt the symmetric choice `uL = -uRL/2`,
`uR = +uRL/2`. Wave speeds and `p*` are Galilean invariants, so HLLC's `p*`
estimate is independent of this choice.

In [3]:
def gas_state_to_primitives(gas_state):
    """(drho, dp, du) -> (rhoL, uL, pL, rhoR, uR, pR) with symmetric frame uL = -uR."""
    drho, dp, du = gas_state
    rhoL = 1.0 - drho
    rhoR = 1.0 + drho
    pL = 1.0 - dp
    pR = 1.0 + dp
    cL = jnp.sqrt(GAMMA * pL / rhoL)
    cR = jnp.sqrt(GAMMA * pR / rhoR)
    ducrit = (2.0 / (GAMMA - 1.0)) * (cL + cR) / jnp.sqrt(GAMMA)
    uRL = ducrit * du
    return rhoL, -0.5 * uRL, pL, rhoR, 0.5 * uRL, pR


def conservatives(rho, u, p):
    E = p / (GAMMA - 1.0) + 0.5 * rho * u * u
    return jnp.array([rho, rho * u, E])


def euler_flux(rho, u, p):
    E = p / (GAMMA - 1.0) + 0.5 * rho * u * u
    return jnp.array([rho * u, rho * u * u + p, u * (E + p)])

## Rusanov (Local Lax-Friedrichs)

Simplest of the bunch: average the fluxes and add a scalar dissipation set by
the maximum acoustic speed at the interface.

$$F = \tfrac{1}{2}(F_L + F_R) - \tfrac{1}{2}\, s_{\max}\,(U_R - U_L),
\quad s_{\max} = \max(|u_L|+c_L,\, |u_R|+c_R).$$

In [4]:
@jax.jit
def rusanov_flux(gas_state):
    rhoL, uL, pL, rhoR, uR, pR = gas_state_to_primitives(gas_state)
    cL = jnp.sqrt(GAMMA * pL / rhoL)
    cR = jnp.sqrt(GAMMA * pR / rhoR)
    smax = jnp.maximum(jnp.abs(uL) + cL, jnp.abs(uR) + cR)
    UL = conservatives(rhoL, uL, pL)
    UR = conservatives(rhoR, uR, pR)
    FL = euler_flux(rhoL, uL, pL)
    FR = euler_flux(rhoR, uR, pR)
    return 0.5 * (FL + FR) - 0.5 * smax * (UR - UL)

## HLL

Collapse the wave fan to two waves with Davis-style speed estimates.

$$S_L = \min(u_L - c_L,\, u_R - c_R), \quad S_R = \max(u_L + c_L,\, u_R + c_R).$$

$$F_{HLL} = \frac{S_R F_L - S_L F_R + S_L S_R (U_R - U_L)}{S_R - S_L}.$$

In [5]:
@jax.jit
def hll_flux(gas_state):
    rhoL, uL, pL, rhoR, uR, pR = gas_state_to_primitives(gas_state)
    cL = jnp.sqrt(GAMMA * pL / rhoL)
    cR = jnp.sqrt(GAMMA * pR / rhoR)
    SL = jnp.minimum(uL - cL, uR - cR)
    SR = jnp.maximum(uL + cL, uR + cR)
    UL = conservatives(rhoL, uL, pL)
    UR = conservatives(rhoR, uR, pR)
    FL = euler_flux(rhoL, uL, pL)
    FR = euler_flux(rhoR, uR, pR)
    F_star = (SR * FL - SL * FR + SL * SR * (UR - UL)) / (SR - SL)
    return jnp.where(SL >= 0, FL, jnp.where(SR <= 0, FR, F_star))

## HLLC

Restore the contact wave inside the HLL fan. We additionally get a clean
star-pressure estimate

$$p_* = p_L + \rho_L (S_L - u_L)(S_* - u_L),$$

where $S_* = (p_R - p_L + \rho_L u_L (S_L - u_L) - \rho_R u_R (S_R - u_R)) /
(\rho_L (S_L - u_L) - \rho_R (S_R - u_R))$ is the contact speed.

In [6]:
def _hllc_core(gas_state):
    rhoL, uL, pL, rhoR, uR, pR = gas_state_to_primitives(gas_state)
    cL = jnp.sqrt(GAMMA * pL / rhoL)
    cR = jnp.sqrt(GAMMA * pR / rhoR)
    SL = jnp.minimum(uL - cL, uR - cR)
    SR = jnp.maximum(uL + cL, uR + cR)

    S_star_num = pR - pL + rhoL * uL * (SL - uL) - rhoR * uR * (SR - uR)
    S_star_den = rhoL * (SL - uL) - rhoR * (SR - uR)
    S_star = S_star_num / S_star_den
    p_star = pL + rhoL * (SL - uL) * (S_star - uL)

    def star_state(rho, u, p, S):
        E = p / (GAMMA - 1.0) + 0.5 * rho * u * u
        factor = rho * (S - u) / (S - S_star)
        return factor * jnp.array([
            1.0,
            S_star,
            E / rho + (S_star - u) * (S_star + p / (rho * (S - u))),
        ])

    UL = conservatives(rhoL, uL, pL)
    UR = conservatives(rhoR, uR, pR)
    FL = euler_flux(rhoL, uL, pL)
    FR = euler_flux(rhoR, uR, pR)
    UL_star = star_state(rhoL, uL, pL, SL)
    UR_star = star_state(rhoR, uR, pR, SR)
    FL_star = FL + SL * (UL_star - UL)
    FR_star = FR + SR * (UR_star - UR)

    F = jnp.where(
        SL >= 0, FL,
        jnp.where(
            S_star >= 0, FL_star,
            jnp.where(SR >= 0, FR_star, FR),
        ),
    )
    return F, p_star


@jax.jit
def hllc_flux(gas_state):
    F, _ = _hllc_core(gas_state)
    return F


@jax.jit
def hllc_pstar(gas_state):
    _, p_star = _hllc_core(gas_state)
    return p_star

## Roe

Linearized Riemann solver about the Roe-averaged state with a Harten
entropy fix on the genuinely-nonlinear fields (kept small, $\varepsilon =
0.1\,\bar c$) so the solver doesn't admit expansion shocks at sonic points.

In [7]:
@jax.jit
def roe_flux(gas_state):
    rhoL, uL, pL, rhoR, uR, pR = gas_state_to_primitives(gas_state)
    HL = (pL / (GAMMA - 1.0) + 0.5 * rhoL * uL * uL + pL) / rhoL
    HR = (pR / (GAMMA - 1.0) + 0.5 * rhoR * uR * uR + pR) / rhoR

    sL, sR = jnp.sqrt(rhoL), jnp.sqrt(rhoR)
    w = 1.0 / (sL + sR)
    rho_a = sL * sR
    u_a = (sL * uL + sR * uR) * w
    H_a = (sL * HL + sR * HR) * w
    c_a = jnp.sqrt((GAMMA - 1.0) * (H_a - 0.5 * u_a * u_a))

    drho = rhoR - rhoL
    du = uR - uL
    dp = pR - pL

    alpha2 = drho - dp / (c_a * c_a)
    alpha1 = 0.5 * (dp - rho_a * c_a * du) / (c_a * c_a)
    alpha3 = 0.5 * (dp + rho_a * c_a * du) / (c_a * c_a)

    lam1 = u_a - c_a
    lam2 = u_a
    lam3 = u_a + c_a

    eps = 0.1 * c_a
    def fix(lam):
        return jnp.where(jnp.abs(lam) < eps, 0.5 * (lam * lam / eps + eps), jnp.abs(lam))

    K1 = jnp.array([1.0, u_a - c_a, H_a - u_a * c_a])
    K2 = jnp.array([1.0, u_a,       0.5 * u_a * u_a])
    K3 = jnp.array([1.0, u_a + c_a, H_a + u_a * c_a])

    FL = euler_flux(rhoL, uL, pL)
    FR = euler_flux(rhoR, uR, pR)
    return 0.5 * (FL + FR) - 0.5 * (
        alpha1 * fix(lam1) * K1
        + alpha2 * fix(lam2) * K2
        + alpha3 * fix(lam3) * K3
    )

## Exact (reference)

The project's iterative Newton+bisection solver from `riemann_pinn.physics`.
This is what we eventually want a network to approximate cheaply.

In [8]:
@jax.jit
def exact_pstar(gas_state):
    p_star, _ = find_pstar(gas_state)
    return p_star

## Sanity check on a Sod-like state

`(drho, dp, du) = (-0.6, -0.6, 0)` corresponds to `pL=1.6, pR=0.4,
rhoL=1.6, rhoR=0.4` (Sod, up to non-dimensionalization). All four
flux solvers should produce visibly similar interface fluxes, and HLLC's
`p*` should agree with the exact iterative solver to a few digits.

In [9]:
sod = jnp.array([-0.6, -0.6, 0.0])
print(f"{'solver':10s}  {'F[mass]':>12s}  {'F[mom]':>12s}  {'F[energy]':>12s}")
for name, fn in [("Rusanov", rusanov_flux), ("HLL", hll_flux), ("HLLC", hllc_flux), ("Roe", roe_flux)]:
    F = fn(sod)
    print(f"{name:10s}  {float(F[0]):12.6f}  {float(F[1]):12.6f}  {float(F[2]):12.6f}")

print(f"\nHLLC  p* = {float(hllc_pstar(sod)):.6f}")
print(f"exact p* = {float(exact_pstar(sod)):.6f}")

solver           F[mass]        F[mom]     F[energy]
Rusanov         0.774597      1.000000      1.161895
HLL             0.774597      1.000000      1.161895
HLLC            0.546774      0.894118      1.038871
Roe             0.480250      1.000000      1.161895

HLLC  p* = 0.640000
exact p* = 0.762058


## Timing

Each solver is `jit`-compiled and `vmap`-ped over a batch of random gas
states drawn from the project's default training domain. We warm up the
compiled function first, then time `n_runs` calls with
`block_until_ready()` to defeat JAX's async dispatch.

In [10]:
batch = 100_000
n_runs = 50

key = jr.PRNGKey(0)
k1, k2, k3 = jr.split(key, 3)
drho = jr.uniform(k1, (batch,), minval=-0.9, maxval=0.9)
dp   = jr.uniform(k2, (batch,), minval=-0.9, maxval=0.9)
du   = jr.uniform(k3, (batch,), minval=-3.0, maxval=0.9)
gas_states = jnp.stack([drho, dp, du], axis=-1)

solvers = {
    "Rusanov":  rusanov_flux,
    "HLL":      hll_flux,
    "HLLC":     hllc_flux,
    "Roe":      roe_flux,
    "exact p*": exact_pstar,
}

print(f"batch = {batch}, runs = {n_runs}")
print(f"{'solver':10s}  {'ms / batch':>12s}  {'ns / state':>12s}")
for name, fn in solvers.items():
    batched = jax.jit(jax.vmap(fn))
    batched(gas_states).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        out = batched(gas_states).block_until_ready()
    dt = (time.perf_counter() - t0) / n_runs
    print(f"{name:10s}  {dt*1e3:12.3f}  {dt/batch*1e9:12.1f}")

batch = 100000, runs = 50
solver        ms / batch    ns / state
Rusanov            0.325           3.3
HLL                0.843           8.4
HLLC               0.834           8.3
Roe                0.675           6.7
exact p*          82.719         827.2
